# menu_items

The present notebook contains the process for data exploration, ingestion and transformation, displayed in a way that allows the reader to get an idea of how the process was done, allowing for better understanding and easier replication.

The processes shown here have been automated, and the scripts are available within the /src folder.

## Stage 0. Dependencies

In [20]:
import os
from dotenv import load_dotenv
import pandas as pd
from sqlalchemy import create_engine
from pathlib import Path
import sys

load_dotenv()

True

## Stage 1. Data exploration

Using `pandas`, it is possible to easily view the data. The tables are stored in the `menu_items.csv` file, located in the `/data` folder.

In [15]:
# Reading the data
data = pd.read_csv("../data/menu_items.csv")

# Displaying the first five rows
data.head()

,item_id,restaurant_id,price
0,M0001,R013,20.34
1,M0002,R090,20.84
2,M0003,R009,15.51
3,M0004,R104,29.88
4,M0005,R003,34.72


From the output of the cell above, it can be inferred that the `menu_items` table is a relational table, which specifies the menu items offered by each restaurant, and the price of each. The data consists of the following:

1. `item_id` (String): Specifies the item.
2. `restaurant_id` (String): Corresponds to the `restaurant_id` in the `restaurants` table.
3. `price` (decimal/float): The price of the specific item at the specific restaurant.

This information was used to create the SQL table, to which the data will be added.

## Stage 2. Data ingestion

Once the data structure has been understood and the SQL table has been created, the data can be ingested by the database. To do that, a connection must be established first.

In [16]:
# Reading environment variables from .env file
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

# Constructing the database URL
DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# Connecting to the database using SQLAlchemy
engine = create_engine(DATABASE_URL)

With the connection established, a new column will be added to the table to specify from which file the data was obtained.

In [17]:
data["source_file"] = "menu_items.csv"
data.head()

,item_id,restaurant_id,price,source_file
0,M0001,R013,20.34,menu_items.csv
1,M0002,R090,20.84,menu_items.csv
2,M0003,R009,15.51,menu_items.csv
3,M0004,R104,29.88,menu_items.csv
4,M0005,R003,34.72,menu_items.csv


Now, the data can be ingested into the database's raw data table.

In [18]:
data.to_sql(
    "menu_items",
    engine,
    schema="raw",
    if_exists="replace",
    index=False
)

400

## Stage 3. Transformation test

Once the data has been ingested into the database, the transformation query can now be tested. First, we connect to the database.

In [23]:
BASE_DIR = Path.cwd().parent
SRC_DIR = BASE_DIR / "src"

sys.path.append(str(SRC_DIR))

from db_connection import get_engine

engine = get_engine()

# Helper function: Returns the result of an SQL query as a DataFrame
from sqlalchemy import text

def run_query(query: str) -> pd.DataFrame:
  with engine.connect() as conn:
    return pd.read_sql_query(text(query), conn)

2026-09-22 11:54:06,524 | INFO | db_connection | Database environment variables validated successfully.
2026-09-22 11:54:06,525 | INFO | db_connection | Building database URL for host=localhost, port=5432, database=hospital_management, user=postgres
2026-09-22 11:54:06,526 | INFO | db_connection | Creating SQLAlchemy engine.


Once connected, we can run a `SELECT` query to view the state of the data.

In [24]:
query = """
SELECT * FROM raw.menu_items;
"""

raw_order_medium_df = run_query(query)
raw_order_medium_df.head()

,item_id,restaurant_id,price,source_file
0,M0001,R013,20.34,menu_items.csv
1,M0002,R090,20.84,menu_items.csv
2,M0003,R009,15.51,menu_items.csv
3,M0004,R104,29.88,menu_items.csv
4,M0005,R003,34.72,menu_items.csv


Finally, we run the transform query, which validates the data and re-formats any invalid values.

In [25]:
transform_query = """
    SELECT
        TRIM(item_id) as item_id,
        TRIM(restaurant_id) as restaurant_id,
        price,
        source_file
    FROM raw.menu_items
    WHERE
        item_id LIKE 'M%'
        AND restaurant_id LIKE 'R%';
  
"""

orders_medium_df = run_query(transform_query)
orders_medium_df.head()

,item_id,restaurant_id,price,source_file
0,M0001,R013,20.34,menu_items.csv
1,M0002,R090,20.84,menu_items.csv
2,M0003,R009,15.51,menu_items.csv
3,M0004,R104,29.88,menu_items.csv
4,M0005,R003,34.72,menu_items.csv


If the query succesfully transforms the data, the code was succesful, and the procedure can now be implemented in SQL.